# Reprise sans retrain

Ce notebook sert a reprendre apres fermeture de VS Code, sans retélécharger le dataset et sans réentraîner le modele.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
DATASET_DIR = DATA_DIR / 'Face4Shifts'
ANNO_DIR = DATASET_DIR / 'Anno'
PHOTOS_DIR = DATASET_DIR / 'Img' / 'Photo'
MODEL_PATH = DATA_DIR / 'model_baseline.pth'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PHOTOS_DIR:', PHOTOS_DIR)
print('MODEL_PATH:', MODEL_PATH)
print('Photos dir exists:', PHOTOS_DIR.exists())
print('Model exists:', MODEL_PATH.exists())

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from PIL import Image

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print('Imports OK')

Imports OK


In [3]:
p_attr_path = ANNO_DIR / 'p_attr.json'
if not p_attr_path.exists():
    raise FileNotFoundError(f'Annotations file not found: {p_attr_path}')

df_photo = pd.read_json(p_attr_path, lines=True)
df_model = df_photo.copy()

sourire_ferme = (df_model['smile_with_closed_lips'] == 1)
sourire_ouvert = (df_model['smile_with_open_lips'] == 1)
condition_sourire = sourire_ferme | sourire_ouvert
df_model['Sourire'] = np.where(condition_sourire, 1, 2)
df_model['Cheveux_Long'] = np.where(df_model['long_hair'] == 1, 1, 2)
df_model['Gender'] = df_model['gender']
df_model['ID'] = df_model['ID'].astype(str).str.strip()

print('Rows:', len(df_model))

Rows: 30000


In [4]:
class FaceDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = str(self.dataframe.loc[idx, 'ID'])
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_name += '.jpg'

        img_path = self.image_dir / img_name
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        sourire = 1.0 if self.dataframe.loc[idx, 'Sourire'] == 1 else 0.0
        gender = 1.0 if self.dataframe.loc[idx, 'Gender'] == 1 else 0.0
        cheveux = 1.0 if self.dataframe.loc[idx, 'Cheveux_Long'] == 1 else 0.0

        return {
            'image': image,
            'target_sourire': torch.tensor(sourire, dtype=torch.float32),
            'sensible_gender': torch.tensor(gender, dtype=torch.float32),
            'proxy_cheveux': torch.tensor(cheveux, dtype=torch.float32)
        }

transformations = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

df_temp, df_test = train_test_split(
    df_model,
    test_size=0.20,
    random_state=42,
    stratify=df_model['Sourire']
)

df_train, df_val = train_test_split(
    df_temp,
    test_size=0.20,
    random_state=42,
    stratify=df_temp['Sourire']
)

test_dataset = FaceDataset(df_test, PHOTOS_DIR, transform=transformations)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print('Test loader ready:', len(test_dataset), 'images')

Test loader ready: 6000 images


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)
model = model.to(device)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model file not found: {MODEL_PATH}. Run training notebook cell 29 then cell 31 once.'
    )

state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Model loaded successfully on', device)

Model loaded successfully on cpu


In [6]:
all_preds = []
all_targets = []
all_genders = []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        outputs = model(images)
        preds = (outputs > 0.0).float().cpu().numpy()
        targets = batch['target_sourire'].cpu().numpy()
        genders = batch['sensible_gender'].cpu().numpy()

        all_preds.extend(preds.flatten())
        all_targets.extend(targets.flatten())
        all_genders.extend(genders.flatten())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_genders = np.array(all_genders)

global_acc = accuracy_score(all_targets, all_preds)
global_precision = precision_score(all_targets, all_preds, zero_division=0)
global_recall = recall_score(all_targets, all_preds, zero_division=0)
global_f1 = f1_score(all_targets, all_preds, zero_division=0)

print('Global test metrics')
print('Accuracy :', round(global_acc, 4))
print('Precision:', round(global_precision, 4))
print('Recall   :', round(global_recall, 4))
print('F1-score :', round(global_f1, 4))

acc_male = accuracy_score(all_targets[all_genders == 1.0], all_preds[all_genders == 1.0])
acc_female = accuracy_score(all_targets[all_genders == 0.0], all_preds[all_genders == 0.0])
disparity = abs(acc_male - acc_female)

print('')
print('Fairness metric')
print('Disparity in Accuracy:', round(disparity, 4))

Global test metrics
Accuracy : 0.8953
Precision: 0.8866
Recall   : 0.9329
F1-score : 0.9091

Fairness metric
Disparity in Accuracy: 0.0353


## Experience 2: cible fusionnee (Sourire ET Cheveux_Long)

Cette experience construit une nouvelle cible qui fusionne sourire et cheveux longs, puis entraine un second modele pour comparer la disparite avec la baseline.

In [7]:
# Construire une cible fusionnee: 1 si (Sourire == 1 et Cheveux_Long == 1), sinon 0
# On garde le meme split pour comparer avec la baseline dans des conditions proches.

df_fusion = df_model.copy()
df_fusion['Target_Fusion'] = np.where(
    (df_fusion['Sourire'] == 1) & (df_fusion['Cheveux_Long'] == 1),
    1,
    0
)

print('Distribution Target_Fusion:')
print(df_fusion['Target_Fusion'].value_counts(normalize=True).rename('ratio').round(4))

class FaceDatasetFusion(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = str(self.dataframe.loc[idx, 'ID'])
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_name += '.jpg'

        img_path = self.image_dir / img_name
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        target = float(self.dataframe.loc[idx, 'Target_Fusion'])
        gender = 1.0 if self.dataframe.loc[idx, 'Gender'] == 1 else 0.0

        return {
            'image': image,
            'target': torch.tensor(target, dtype=torch.float32),
            'sensible_gender': torch.tensor(gender, dtype=torch.float32)
        }

# Split stratifie sur la cible fusionnee
df_temp_f, df_test_f = train_test_split(
    df_fusion,
    test_size=0.20,
    random_state=42,
    stratify=df_fusion['Target_Fusion']
)

df_train_f, df_val_f = train_test_split(
    df_temp_f,
    test_size=0.20,
    random_state=42,
    stratify=df_temp_f['Target_Fusion']
)

train_fusion = FaceDatasetFusion(df_train_f, PHOTOS_DIR, transform=transformations)
val_fusion = FaceDatasetFusion(df_val_f, PHOTOS_DIR, transform=transformations)
test_fusion = FaceDatasetFusion(df_test_f, PHOTOS_DIR, transform=transformations)

train_loader_f = DataLoader(train_fusion, batch_size=32, shuffle=True, num_workers=0)
val_loader_f = DataLoader(val_fusion, batch_size=32, shuffle=False, num_workers=0)
test_loader_f = DataLoader(test_fusion, batch_size=32, shuffle=False, num_workers=0)

# Modele 2 (meme architecture)
model_f = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs_f = model_f.fc.in_features
model_f.fc = nn.Linear(num_ftrs_f, 1)
model_f = model_f.to(device)

criterion_f = nn.BCEWithLogitsLoss()
optimizer_f = torch.optim.Adam(model_f.parameters(), lr=1e-4)

num_epochs_f = 2
print('\nDebut entrainement modele fusion...')
for epoch in range(num_epochs_f):
    model_f.train()
    train_loss_f = 0.0
    for batch in train_loader_f:
        images = batch['image'].to(device)
        targets = batch['target'].unsqueeze(1).to(device)

        optimizer_f.zero_grad()
        outputs = model_f(images)
        loss = criterion_f(outputs, targets)
        loss.backward()
        optimizer_f.step()

        train_loss_f += loss.item()

    model_f.eval()
    val_loss_f = 0.0
    with torch.no_grad():
        for batch in val_loader_f:
            images = batch['image'].to(device)
            targets = batch['target'].unsqueeze(1).to(device)
            outputs = model_f(images)
            val_loss_f += criterion_f(outputs, targets).item()

    print(f"Epoch {epoch+1}/{num_epochs_f} | train_loss={train_loss_f/len(train_loader_f):.4f} | val_loss={val_loss_f/len(val_loader_f):.4f}")

# Evaluation du modele fusion
model_f.eval()
preds_f, targets_f, genders_f = [], [], []
with torch.no_grad():
    for batch in test_loader_f:
        images = batch['image'].to(device)
        outputs = model_f(images)
        pred = (outputs > 0.0).float().cpu().numpy().flatten()
        tgt = batch['target'].cpu().numpy().flatten()
        gen = batch['sensible_gender'].cpu().numpy().flatten()

        preds_f.extend(pred)
        targets_f.extend(tgt)
        genders_f.extend(gen)

preds_f = np.array(preds_f)
targets_f = np.array(targets_f)
genders_f = np.array(genders_f)

acc_fusion = accuracy_score(targets_f, preds_f)
acc_fusion_m = accuracy_score(targets_f[genders_f == 1.0], preds_f[genders_f == 1.0])
acc_fusion_f = accuracy_score(targets_f[genders_f == 0.0], preds_f[genders_f == 0.0])
disparity_fusion = abs(acc_fusion_m - acc_fusion_f)

# Comparaison avec la baseline (disparity calculee plus haut)
print('\nComparaison Equite')
print(f'Baseline disparity (Sourire): {disparity:.4f}')
print(f'Fusion disparity (Sourire&Cheveux): {disparity_fusion:.4f}')
print(f'Baseline accuracy: {global_acc:.4f}')
print(f'Fusion accuracy: {acc_fusion:.4f}')

if disparity_fusion > disparity:
    print('Resultat: la cible fusionnee augmente le biais mesure.')
elif disparity_fusion < disparity:
    print('Resultat: la cible fusionnee reduit le biais mesure.')
else:
    print('Resultat: biais similaire entre baseline et fusion.')

Distribution Target_Fusion:
Target_Fusion
0    0.7023
1    0.2977
Name: ratio, dtype: float64

Debut entrainement modele fusion...
Epoch 1/2 | train_loss=0.2651 | val_loss=0.2099
Epoch 2/2 | train_loss=0.1648 | val_loss=0.2255

Comparaison Equite
Baseline disparity (Sourire): 0.0353
Fusion disparity (Sourire&Cheveux): 0.1310
Baseline accuracy: 0.8953
Fusion accuracy: 0.9085
Resultat: la cible fusionnee augmente le biais mesure.


## Sauvegarder les resultats pour reprise rapide

Ces cellules permettent d'enregistrer les resultats de la baseline et de l'experience fusionnee, puis de les recharger sans relancer les modeles.

In [8]:
import json

RESULTS_DIR = DATA_DIR / 'results_cache'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Sauvegarde baseline
baseline_payload = {
    'global_acc': float(global_acc),
    'global_precision': float(global_precision),
    'global_recall': float(global_recall),
    'global_f1': float(global_f1),
    'disparity': float(disparity)
}

with open(RESULTS_DIR / 'baseline_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_payload, f, indent=2)

np.save(RESULTS_DIR / 'baseline_preds.npy', all_preds)
np.save(RESULTS_DIR / 'baseline_targets.npy', all_targets)
np.save(RESULTS_DIR / 'baseline_genders.npy', all_genders)

# Sauvegarde fusion (si disponible)
fusion_available = all(name in globals() for name in ['acc_fusion', 'disparity_fusion', 'preds_f', 'targets_f', 'genders_f'])
if fusion_available:
    fusion_payload = {
        'acc_fusion': float(acc_fusion),
        'disparity_fusion': float(disparity_fusion)
    }
    with open(RESULTS_DIR / 'fusion_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(fusion_payload, f, indent=2)

    np.save(RESULTS_DIR / 'fusion_preds.npy', preds_f)
    np.save(RESULTS_DIR / 'fusion_targets.npy', targets_f)
    np.save(RESULTS_DIR / 'fusion_genders.npy', genders_f)

# Tableau resume
summary = {
    'baseline_acc': float(global_acc),
    'baseline_disparity': float(disparity),
    'fusion_acc': float(acc_fusion) if fusion_available else None,
    'fusion_disparity': float(disparity_fusion) if fusion_available else None
}
summary_df = pd.DataFrame([summary])
summary_df.to_csv(RESULTS_DIR / 'summary_metrics.csv', index=False)

print('Resultats sauvegardes dans:', RESULTS_DIR)
print('Fichiers: baseline_metrics.json, summary_metrics.csv, *.npy')
if fusion_available:
    print('Fichiers fusion egalement sauvegardes.')
else:
    print('Fusion non detectee: execute d abord la cellule experience fusionnee.')

Resultats sauvegardes dans: /home/anfau/projets/fair/data/results_cache
Fichiers: baseline_metrics.json, summary_metrics.csv, *.npy
Fichiers fusion egalement sauvegardes.


In [9]:
# Rechargement rapide des resultats (sans relancer les modeles)
import json

RESULTS_DIR = DATA_DIR / 'results_cache'

baseline_path = RESULTS_DIR / 'baseline_metrics.json'
summary_path = RESULTS_DIR / 'summary_metrics.csv'

if not baseline_path.exists():
    raise FileNotFoundError(f'Resultats non trouves: {baseline_path}. Lance d abord la cellule de sauvegarde.')

with open(baseline_path, 'r', encoding='utf-8') as f:
    baseline_metrics = json.load(f)

print('Baseline rechargee:')
for k, v in baseline_metrics.items():
    print(f'  {k}: {v:.4f}')

if summary_path.exists():
    print('\nResume compare (CSV):')
    display(pd.read_csv(summary_path))
else:
    print('\nPas de summary_metrics.csv trouve.')

Baseline rechargee:
  global_acc: 0.8953
  global_precision: 0.8866
  global_recall: 0.9329
  global_f1: 0.9091
  disparity: 0.0353

Resume compare (CSV):


,baseline_acc,baseline_disparity,fusion_acc,fusion_disparity
0,0.895333,0.035315,0.9085,0.130972
